**History Lesson Abstract**

Goal: Summarize a long article about WWII into a flashcard format.

Tech: Summarization.

We begin by installing and importing the Hugging Face transformers library, which provides pretrained NLP models and the high-level pipeline() API.

In [1]:
!pip install transformers


In [2]:
from transformers import pipeline


In [19]:
!pip install gradio


We load the BART Large CNN model, which is fine-tuned for abstractive text summarization, making it suitable for summarizing long historical articles.

In [3]:
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


The summarizer converts the long text into a shorter, meaningful summary while preserving key historical facts.The summarized text is split into short points, creating flashcard-style outputs for easier learning. This also has 'Ending-word filter' which helps to removes any sentence that was cut off.

In [37]:
def summarize_to_flashcards(text):
    paragraphs = [p for p in text.split("\n") if p.strip()]
    all_flashcards = []
    card_num = 1

    for para in paragraphs:
        summary = summarizer(
            para,
            max_length=120,
            min_length=50,
            do_sample=False
        )[0]["summary_text"]

        sentences = [s.strip() for s in summary.split(". ") if s.strip()]

        for s in sentences:
            words = s.split()

            # Filter incomplete or cut-off sentences
            if len(words) < 6:
                continue
            if s.endswith(("of", "to", "by", "and", "with", "for")):
                continue

            all_flashcards.append(
                f"🗂️ Flashcard {card_num}\n"
                f"➡️ {s}.\n"
                + "-" * 50
            )
            card_num += 1

    return "📘 History Lesson Flashcards\n\n" + "\n".join(all_flashcards)


This helps us with a simple UI where we can add a paragraph and we would get the flashcards as output.

In [38]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("# 📘 History Lesson Abstract Generator")
    gr.Markdown(
        "Paste a history article below to generate flashcard-style summaries using Generative AI."
    )

    input_text = gr.Textbox(
        lines=18,
        placeholder="Paste history text here..."
    )

    output_text = gr.Textbox(
        lines=22,
        label="Flashcard Summary"
    )

    generate_btn = gr.Button("Generate Flashcards")

    generate_btn.click(
        fn=summarize_to_flashcards,
        inputs=input_text,
        outputs=output_text
    )

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://464fb29fa7037ea69a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
